In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

columns = ["Rank", "Country", "Total Earnings", "Total Number of Players", "Most Popular Game", "Earnings from the Game"]

def get_country_details(row):
    cells = row.find_elements(By.TAG_NAME, "td")
    texts = [cell.text.strip() for cell in cells]

    # Remove texts after numbers in columns
    def clean_stat(text):
        return (
            text.replace("Player", "")
                .replace("Players", "")
                .replace("s", "")                   # Had to remove the "s" characters too since player and players are distinct texts
                .replace("\n", " ")
                .strip()
        )


    return {
        "Rank": texts[0], 
        "Country": texts[1], 
        "Total Earnings": texts[2],              
        "Total Number of Players": clean_stat(texts[3]),
        "Most Popular Game": texts[4],  
        "Earnings from the Game": texts[5],
        
        
        
    }

def main():
    country_data = []

    driver = webdriver.Chrome()

    try:
        for page_id in range(1, 5):
            url = f"https://www.esportsearnings.com/countries?page={page_id}"
            driver.get(url)

            WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
            )

            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

            for row in rows:
                country = get_country_details(row)
                if country:
                    country_data.append(country)

                if len(country_data) >= 100:
                    break

            if len(country_data) >= 100:
                break

            for row in rows[:10]:
                cells = row.find_elements(By.TAG_NAME, "td")
                print(len(cells), [c.text for c in cells])


    finally:
        driver.quit()

    df = pd.DataFrame(country_data[:100], columns=columns)
    df.to_csv("top_100_countries_by_prize_pool.csv", index=False)

    print(df.head())
    print(f"Saved {len(df)} countries.")

if __name__ == "__main__":
    main()

  Rank             Country   Total Earnings Total Number of Players  \
0   1.               China  $332,711,178.23                   9,371   
1   2.       United States  $300,867,001.46                  29,445   
2   3.               Korea  $157,079,924.92                   5,986   
3   4.  Russian Federation   $97,092,081.59                   5,875   
4   5.              Brazil   $73,016,255.76                   6,031   

   Most Popular Game Earnings from the Game  
0             Dota 2         $86,844,959.84  
1           Fortnite         $52,378,533.18  
2  League of Legends         $41,645,617.74  
3             Dota 2         $43,417,776.36  
4  Rainbow Six Siege         $13,311,887.49  
Saved 100 countries.
